# Phase 1 — Data Acquisition and Ingestion
This notebook downloads Men's IPL and T20Is match data from Cricsheet, parses basic metadata, creates a match index, and produces initial data distribution plots.


In [ ]:
import os
import sys
import urllib.request
import zipfile
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

sys.path.append(os.path.abspath(os.path.join('..')))
from src.common.config import RAW_DATA_DIR, INTERIM_DATA_DIR, REPORTS_DIR, FIGURES_DIR, CRICSHEET_IPL_URL, CRICSHEET_T20I_URL


## 1. Download Datasets
We retrieve Men's IPL and T20Is JSON zip archives from Cricsheet.


In [ ]:
ipl_zip_path = os.path.join(RAW_DATA_DIR, 'ipl_male_json.zip')
t20i_zip_path = os.path.join(RAW_DATA_DIR, 't20is_male_json.zip')

if not os.path.exists(ipl_zip_path):
    print('Downloading IPL male JSON data...')
    urllib.request.urlretrieve(CRICSHEET_IPL_URL, ipl_zip_path)
    print('Downloaded IPL zip.')
else:
    print('IPL zip already exists.')

if not os.path.exists(t20i_zip_path):
    print('Downloading T20I male JSON data...')
    urllib.request.urlretrieve(CRICSHEET_T20I_URL, t20i_zip_path)
    print('Downloaded T20I zip.')
else:
    print('T20I zip already exists.')


## 2. Parse Match Index
We iterate through the JSON files in each zip to construct an index of matches, checking for DLS application and basic attributes.


In [ ]:
def parse_zip_metadata(zip_path, competition_name):
    metadata_list = []
    with zipfile.ZipFile(zip_path, 'r') as archive:
        file_list = [f for f in archive.namelist() if f.endswith('.json') and not f.startswith('README')]
        print(f'Parsing {len(file_list)} matches from {competition_name} zip...')
        for file_name in file_list:
            try:
                with archive.open(file_name) as f:
                    data = json.load(f)
                    info = data.get('info', {})
                    
                    dates = info.get('dates', [])
                    match_date = dates[0] if dates else None
                    
                    outcome = info.get('outcome', {})
                    method = outcome.get('method')
                    has_dls = method in ['D/L', 'DLS']
                    
                    revised_target = None
                    innings = data.get('innings', [])
                    for inn in innings:
                        if 'target' in inn:
                            revised_target = inn['target'].get('runs')
                            has_dls = True
                    
                    teams = info.get('teams', [])
                    team1 = teams[0] if len(teams) > 0 else None
                    team2 = teams[1] if len(teams) > 1 else None
                    
                    match_id = os.path.splitext(os.path.basename(file_name))[0]
                    
                    metadata_list.append({
                        'match_id': match_id,
                        'competition': competition_name,
                        'date': match_date,
                        'venue': info.get('venue'),
                        'city': info.get('city'),
                        'team1': team1,
                        'team2': team2,
                        'toss_winner': info.get('toss', {}).get('winner'),
                        'toss_decision': info.get('toss', {}).get('decision'),
                        'outcome_method': method,
                        'has_dls_applied': has_dls,
                        'overs': info.get('overs'),
                        'dls_revised_target': revised_target
                    })
            except Exception as e:
                print(f'Error parsing {file_name}: {e}')
    return metadata_list

ipl_meta = parse_zip_metadata(ipl_zip_path, 'IPL')
t20i_meta = parse_zip_metadata(t20i_zip_path, 'T20I')

df_meta = pd.DataFrame(ipl_meta + t20i_meta)
df_meta['date'] = pd.to_datetime(df_meta['date'])
df_meta.to_parquet(os.path.join(INTERIM_DATA_DIR, 'match_index.parquet'), index=False)
print(f'Saved match index with {len(df_meta)} records.')


## 3. Visualizations and Data Summary


In [ ]:
df_meta = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, 'match_index.parquet'))
df_meta['year'] = df_meta['date'].dt.year

plt.figure(figsize=(10, 5))
sns.countplot(data=df_meta, x='year', hue='competition')
plt.title('Number of Matches per Year and Competition')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'matches_per_year.png'))
plt.show()

plt.figure(figsize=(6, 4))
sns.countplot(data=df_meta, x='has_dls_applied')
plt.title('Matches with DLS Applied')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dls_applied_count.png'))
plt.show()

venue_counts = df_meta['venue'].value_counts().head(20).reset_index()
venue_counts.columns = ['venue', 'match_count']
plt.figure(figsize=(12, 6))
sns.barplot(data=venue_counts, y='venue', x='match_count', palette='viridis')
plt.title('Top 20 Venues by Match Count')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'matches_per_venue.png'))
plt.show()

print(f'Total matches: {len(df_meta)}')
print(f'DLS matches count: {df_meta["has_dls_applied"].sum()}')


## Summary & Conclusion
In this notebook, we downloaded the Cricsheet match files for the IPL and T20Is, parsed their info headers, and saved the basic match index to a parquet table. DLS-applied matches have been flagged, and their scarcity is noted.
